In [5]:
import requests
import fitz
import re
import pandas as pd
from io import BytesIO
import google.auth
from google.auth.transport.requests import AuthorizedSession
from google.auth.exceptions import RefreshError
import pandas as pd

In [16]:


# ETFs BlackRock
ETFS = {
    "IU29": {
        "isin": "IE00094KNQN4",
        "url": "https://www.blackrock.com/uk/literature/fact-sheet/iu29-ishares-ibonds-dec-2029-term-high-yield-corp-ucits-etf-fund-fact-sheet-en-gb.pdf"
    },
    "IT29": {
        "isin": "IE00055OM853",
        "url": "https://www.blackrock.com/uk/literature/fact-sheet/it29-ishares-ibonds-dec-2029-term-treasury-ucits-etf-fund-fact-sheet-en-gb.pdf"
    }
}

def extraire_indicateur(texte, indicateur):
    pattern = rf"{re.escape(indicateur)}\s*:\s*([0-9]+(?:[.,][0-9]+)?)"
    match = re.search(pattern, texte, flags=re.IGNORECASE)

    if not match:
        raise ValueError(f"Indicateur introuvable : {indicateur}")

    return float(match.group(1).replace(",", "."))


def recuperer_blackrock(etf, informations):

    response = requests.get(
        informations["url"],
        headers={"User-Agent": "Mozilla/5.0"},
        timeout=30
    )
    response.raise_for_status()

    pdf = fitz.open(stream=response.content, filetype="pdf")

    try:
        texte = pdf[0].get_text(sort=True)
    finally:
        pdf.close()

    # Vérification de l'ISIN
    if informations["isin"] not in texte:
        raise ValueError(f"ISIN non trouvé pour {etf}")

    # Date officielle des données
    match_date = re.search(
        r"All other data as at\s*:?\s*(\d{1,2}-[A-Za-z]{3}-\d{4})",
        texte,
        flags=re.IGNORECASE
    )

    if not match_date:
        raise ValueError(f"Date BlackRock introuvable pour {etf}")

    date_blackrock = pd.to_datetime(
        match_date.group(1),
        format="%d-%b-%Y"
    ).date()

    return {
        "ETF": etf,
        "ISIN": informations["isin"],
        "Date BlackRock": date_blackrock,
        "Yield to Worst": extraire_indicateur(texte, "Yield to Worst"),
        "Effective Duration": extraire_indicateur(texte, "Effective Duration"),
        "Number of Holdings": int(
            extraire_indicateur(texte, "Number of Holdings")
        )
    }


# Récupération des deux ETFs
resultats = [
    recuperer_blackrock(etf, informations)
    for etf, informations in ETFS.items()
]

# DataFrame utilisé par la cellule Google Sheets
tableau = pd.DataFrame(resultats)

display(tableau)

# Calcul du spread IU29 - IT29 en points de base
ytw = tableau.set_index("ETF")["Yield to Worst"]
spread = (ytw["IU29"] - ytw["IT29"]) * 100

print(f"\nSpread IU29 - IT29 : {spread:.0f} pb")



,ETF,ISIN,Date BlackRock,Yield to Worst,Effective Duration,Number of Holdings
0,IU29,IE00094KNQN4,2026-10-05,8.18,2.09,274
1,IT29,IE00055OM853,2026-10-05,4.96,2.51,37



Spread IU29 - IT29 : 322 pb


In [17]:
import pandas as pd
import google.auth
from google.auth.transport.requests import AuthorizedSession
from urllib.parse import quote

# Configuration
SHEET_ID = "1uM95hnmBiMAlGpkJ2VWrMiR0acqBdcTQ0uWsKkTM8hQ"
ONGLET = "Historique_Ibonds"

# Authentification Google existante
credentials, project = google.auth.default(
    scopes=["https://www.googleapis.com/auth/spreadsheets"]
)
session = AuthorizedSession(credentials)

BASE_URL = (
    f"https://sheets.googleapis.com/v4/spreadsheets/"
    f"{SHEET_ID}/values"
)

def enregistrer_blackrock(tableau):

    df = tableau.set_index("ETF")
    iu29 = df.loc["IU29"]
    it29 = df.loc["IT29"]

    date_iu29 = pd.to_datetime(iu29["Date BlackRock"]).date()
    date_it29 = pd.to_datetime(it29["Date BlackRock"]).date()

    if date_iu29 != date_it29:
        raise ValueError("Les dates BlackRock sont différentes.")

    date = date_iu29.isoformat()

    # Lecture des dates existantes
    plage_lecture = quote(f"'{ONGLET}'!A2:A", safe="")
    r = session.get(f"{BASE_URL}/{plage_lecture}")
    r.raise_for_status()

    dates_existantes = {
        pd.to_datetime(ligne[0], dayfirst=False).date().isoformat()
        for ligne in r.json().get("values", [])
        if ligne and ligne[0].strip()
    }

    # Éviter les doublons
    if date in dates_existantes:
        print(f"{date} : déjà présent, aucune ligne ajoutée.")
        return

    # Données à enregistrer
    ytw_iu29 = float(iu29["Yield to Worst"])
    duration_iu29 = float(iu29["Effective Duration"])
    ytw_it29 = float(it29["Yield to Worst"])
    duration_it29 = float(it29["Effective Duration"])

    spread = round((ytw_iu29 - ytw_it29) * 100, 2)

    valeurs = [[
        date,
        ytw_iu29,
        duration_iu29,
        ytw_it29,
        duration_it29,
        spread
    ]]

    # Écriture Google Sheets
    plage_ecriture = quote(f"'{ONGLET}'!A:F", safe="")

    r = session.post(
        f"{BASE_URL}/{plage_ecriture}:append",
        params={
            "valueInputOption": "RAW",
            "insertDataOption": "INSERT_ROWS"
        },
        json={"values": valeurs}
    )

    print("HTTP :", r.status_code)
    if not r.ok:
        print("Erreur Google :", r.text[:2000])
    r.raise_for_status()

    print(f"Observation du {date} ajoutée.")
    print(f"Spread IU29–IT29 : {spread:.0f} pb")


# Exécution
enregistrer_blackrock(tableau)

HTTP : 200
Observation du 2026-10-05 ajoutée.
Spread IU29–IT29 : 322 pb
